<a href="https://colab.research.google.com/github/diazcas2/mIArmario/blob/main/main0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import sys
!pip install google-search-results --quiet
!pip install google-genai --quiet

# Instala numpy y scipy a versiones compatibles para evitar el ImportError de '_center'.
# Esto debe hacerse antes de transformers que tiene scipy como dependencia.
# Se utiliza --force-reinstall para asegurar que estas versiones sean las que se usan.
!pip install numpy==1.26.0 scipy==1.11.4 --force-reinstall --quiet

# Instala Pillow a una versión que satisfaga rembg (>=12.1.0) y sea compatible con otras librerías.
# Pillow 9.5.0 es compatible con torchvision y debería funcionar con las demás librerías.
!pip install Pillow==9.5.0

# Ahora instala las librerías principales. Deberían respetar la versión de Pillow ya instalada.
!pip install torch transformers google-generativeai langdetect --quiet
!pip install torchvision --quiet # Instala torchvision explícitamente
!pip install huggingface_hub
!pip install -q gradio_client

!pip install qwen-vl-utils -q

!pip install "rembg[cpu]" onnxruntime -q
!apt-get install -y fonts-dejavu -q

!pip install beautifulsoup4 -q

# Reiniciar el entorno de ejecución (Runtime) después de este cambio y ejecutar todo de nuevo.

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
rembg 2.0.73 requires numpy<3.0.0,>=2.3.0, but you have numpy 1.26.0 which is incompatible.
rembg 2.0.73 requires scipy<2.0.0,>=1.16.3, but you have scipy 1.11.4 which is incompatible.
music21 9.9.1 requires numpy>=1.26.4, but you have numpy 1.26.0 which is incompatible.
opencv-contrib-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.0 which is incompatible.
pytensor 2.38.2 requires numpy>=2.0, but you have numpy 1.26.0 which is incompatible.
giddy 2.3.8 requires scipy>=1.12, but you have scipy 1.11.4 which is incompatible.
libpysal 4.14.1 requires scipy>=1.12.0, but you have scipy 1.11.4 which is incompatible.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.0 which is incompatible.
tobler 0.13.0 requires numpy>=2.0, but you have numpy

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/base_command.py", line 179, in exc_logging_wrapper
    status = run_func(*args)
             ^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/req_command.py", line 67, in wrapper
    return func(self, options, args)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/commands/install.py", line 447, in run
    conflicts = self._determine_conflicts(to_install)
                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/commands/install.py", line 578, in _determine_conflicts
    return check_install_conflicts(to_install)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/operations/check.py", line 101, in check_install_conflicts
^C


In [14]:
import json
import re
import time
import torch
import google.generativeai as genai
from transformers import pipeline
from google.colab import userdata
from serpapi import GoogleSearch
import requests
import os

from transformers import WhisperProcessor, WhisperForConditionalGeneration
from IPython.display import display, Image as IPImage


from google.colab import drive
drive.mount('/content/drive')

import sys
sys.path.append('/content/drive/MyDrive/Colab Notebooks/MIARMARIO/')

from Modulo1 import *
from Modulo30 import *
from Modulo2 import *
from Modulo40 import *


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [15]:
processor = WhisperProcessor.from_pretrained("openai/whisper-large-v3")
whisper = WhisperForConditionalGeneration.from_pretrained("openai/whisper-large-v3",torch_dtype=torch.float32 )
device = "cuda" if torch.cuda.is_available() else "cpu"
whisper_model = whisper.to(device)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/1259 [00:00<?, ?it/s]

In [20]:
GEMINI_API_KEY=userdata.get('GEMINI_API_KEY')
GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')   # Google Cloud → Custom Search API
SEARCH_ENGINE_ID = userdata.get('SEARCH_ENGINE_ID') # programmablesearchengine.google.com → cx

SERPAPI_KEY = userdata.get('SERPAPI_KEY')  # serpapi.com → dashboard → API Key

CARPETA_ROPA      = "/content/drive/MyDrive/Colab Notebooks/MIARMARIO/ARMARIO"          # Carpeta con fotos del armario
RUTA_ARMARIO_JSON = "/content/drive/MyDrive/Colab Notebooks/MIARMARIO/armario.json"  # BD persistente de prendas

CARPETA_SALIDA    = '/content/drive/MyDrive/Colab Notebooks/MIARMARIO/CARPETA SALIDA'

genai.configure(api_key=GEMINI_API_KEY)
model = genai.GenerativeModel("gemini-2.5-flash")

# Paso 1. Cargar armario.

In [21]:
'''if forzar_reconstruccion and os.path.exists(RUTA_ARMARIO_JSON):
        os.remove(RUTA_ARMARIO_JSON)
        print("[Paso 1] Armario eliminado para reconstrucción completa")'''

if os.path.exists(RUTA_ARMARIO_JSON):
    print("[Paso 1] Usando armario.json existente.")
    armario = cargar_armario(RUTA_ARMARIO_JSON)
elif os.path.exists(CARPETA_ROPA):
    print("[Paso 1] Construyendo armario desde las fotos...")
    armario = modulo2_construir_armario(
        carpeta_ropa=CARPETA_ROPA,
        ruta_armario_json=RUTA_ARMARIO_JSON,
    )
else:
    raise FileNotFoundError("No hay armario.json ni carpeta ARMARIO. Añade fotos para continuar.")


[Paso 1] Usando armario.json existente.


In [22]:
print(json.dumps(armario, ensure_ascii=False, indent=2))

[
  {
    "tipo": "chaqueta",
    "color": "azul",
    "formalidad": "casual",
    "temporada": [
      "invierno"
    ],
    "id": "prenda_001_1",
    "imagen_path": "/content/drive/MyDrive/Colab Notebooks/MIARMARIO/ARMARIO/abrigo.jpg"
  },
  {
    "tipo": "pantalón",
    "color": "negro",
    "formalidad": "casual",
    "temporada": [
      "invierno",
      "primavera"
    ],
    "id": "prenda_002_1",
    "imagen_path": "/content/drive/MyDrive/Colab Notebooks/MIARMARIO/ARMARIO/pantalontraje.png"
  },
  {
    "tipo": "camisa",
    "color": "gris",
    "estampado": "liso",
    "formalidad": "formal",
    "temporada": [
      "primavera",
      "verano"
    ],
    "id": "prenda_003_1",
    "imagen_path": "/content/drive/MyDrive/Colab Notebooks/MIARMARIO/ARMARIO/polo.webp"
  },
  {
    "tipo": "camisa",
    "color": "azul",
    "estampado": "liso",
    "formalidad": "formal",
    "temporada": [
      "primavera",
      "verano"
    ],
    "id": "prenda_004_1",
    "imagen_path": "/conte

# Paso 2. Procesar instrucciones del ususario.


In [23]:
resultado_mod1 = modulo1_entrada(
    fuente="texto",
    texto="hola quiero que me digas un outfit que sea bueno para irme de escalada por el dia que hace calor",
    ciudad="Cabra",
    model=model
)



Límite alcanzado, esperando 35s...


KeyboardInterrupt: 

In [ ]:
#a mano por si no quedan consultas
resultado_mod1 = {
  "texto": "Quiero un outfit para una cena esta noche. Mi jefe viene, así que necesito ir arreglado.",
  "fuente": "texto",
  "idioma": "es",
  "contexto": {
    "ciudad": "Sevilla",
    "temperatura": 19.2,
    "unidad": "celsius"
  }
}


In [ ]:
print(json.dumps(resultado_mod1, ensure_ascii=False, indent=2))

In [ ]:
# Ejemplo 2 — Entrada por voz
resultado_mod1 = modulo1_entrada(
    fuente="voz",
    ruta_audio="/content/drive/MyDrive/Colab Notebooks/MIARMARIO/mi_audio.mp4",# <-- cambia por tu archivo
    ciudad="Sevilla",
    model=model,
    whisper=whisper,
    processor=processor,
    device=device
)



In [8]:
print(json.dumps(resultado_mod1, ensure_ascii=False, indent=2))

{
  "texto": "Conjunto para escalada de día con calor.",
  "fuente": "texto",
  "idioma": "es",
  "contexto": {
    "ciudad": "Cabra",
    "temperatura": 17.9,
    "unidad": "celsius"
  }
}


# Paso 3. Seleccionar outfit.

In [9]:
'''armario = {
    "armario": [
        {
            "id": "prenda_001",
            "tipo": "camisa",
            "color": "blanco",
            "estampado": "liso",
            "material": "algodón",
            "formalidad": "formal",
            "temporada": ["primavera", "verano", "otoño"],
            "imagen_path": "fotos/camisa_blanca_001.jpg"
        },
        {
            "id": "prenda_002",
            "tipo": "pantalón",
            "color": "negro",
            "estampado": "liso",
            "material": "lana",
            "formalidad": "formal",
            "temporada": ["otoño", "invierno"],
            "imagen_path": "fotos/pantalon_negro_002.jpg"
        }
    ]
}'''


resultado_mod3 = modulo3_seleccion_outfit(
    armario_json=armario,
    instruccion_usuario=resultado_mod1["texto"],
    contexto=resultado_mod1["contexto"],
    model=model,
    SERPAPI_KEY=SERPAPI_KEY
)



[Módulo 3] Outfits posibles detectados: 5
[Módulo 3] Queries generadas: ['pantalón corto escalada negro zalando', 'pies gato escalada hombre zalando', 'camiseta transpirable azul pull']
[Módulo 3] Resultados de tiendas encontrados: 15
[Módulo 3] Éxito: 5 outfits generados.


In [10]:
print(json.dumps(resultado_mod3, ensure_ascii=False, indent=2))

{
  "outfits": [
    {
      "nombre": "Escalada Ligera y Transpirable",
      "prendas": [
        {
          "id": "prenda_010_1",
          "tipo": "camiseta",
          "color": "verde"
        },
        {
          "id": "prenda_018_1",
          "tipo": "shorts",
          "color": "gris"
        }
      ],
      "estilo": "sport",
      "ocasion": "Escalada de día con calor",
      "temperatura": 17.9,
      "prompt_imagen": "male climber wearing a green running t-shirt, grey athletic shorts, and specialized climbing shoes, outdoor rock climbing wall, sunny day, active pose, full body, fashion photography, studio lighting",
      "referencias_tiendas": [
        {
          "prenda": "Pies de gato deportivos primavera-verano de hombre",
          "tienda": "https://www.zalando.es › pies-de-gato-deporte-hombre",
          "enlace": "https://www.zalando.es/pies-de-gato-deporte-hombre/?season_name=primavera-verano",
          "imagen": "",
          "por_que": "Artículo esencial 

In [ ]:
#resultado por si no quedan consultas

resultado_mod3 = {
  "outfit": {
    "prendas": [
      {
        "id": "prenda_004_1",
        "tipo": "camisa",
        "color": "azul"
      },
      {
        "id": "prenda_002_1",
        "tipo": "pantalón",
        "color": "negro"
      },
      {
        "id": "prenda_008_1",
        "tipo": "zapatos",
        "color": "marrón"
      },
      {
        "id": "prenda_019_1",
        "tipo": "cinturón",
        "color": "marrón oscuro"
      },
      {
        "id": "prenda_021_1",
        "tipo": "reloj",
        "color": "negro"
      }
    ],
    "estilo": "formal",
    "ocasion": "cena",
    "temperatura": 18
  },
  "prompt_imagen": "A man wearing a formal plain blue button-up shirt, black formal trousers, dark brown leather penny loafers, a dark brown plain leather belt, and a minimalist black watch. Full body, fashion photography, studio lighting, elegant, professional, confident pose.",
  "referencias_tiendas": [
    {
      "prenda": "Corbatas y pajaritas de hombre en color azul",
      "tienda": "Zalando.es",
      "enlace": "https://www.zalando.es/corbatas-pajaritas-hombre/_azul/",
      "por_que": "Para añadir un toque de sofisticación a la camisa azul y elevar aún más la formalidad para una cena de empresa."
    },
    {
      "prenda": "Zapatos Formales Hombre | ZARA España",
      "tienda": "https://www.zara.com › ... › COLECCIÓN › ZAPATOS",
      "enlace": "https://www.zara.com/es/es/hombre-zapatos-formal-l2568.html",
      "por_que": "Una opción de calzado formal en negro para aquellos que prefieran un look más clásico y monocromático que los mocasines marrones."
    }
  ]
}


print(json.dumps(resultado_mod3, ensure_ascii=False, indent=2))

In [24]:
ruta_lookbook = modulo4_generar_imagen(
    resultado_mod3 = resultado_mod3,
    armario        = armario,
    carpeta_salida = CARPETA_SALIDA,
    model          = model,
)

display(IPImage(filename=ruta_lookbook))
print(f'Lookbook guardado en: {ruta_lookbook}')

[Módulo 4] Generando lookbook con 5 outfit(s)...


Límite alcanzado, esperando 35s...


KeyboardInterrupt: 

In [ ]:
for p in armario:
    print(p["id"], "→", p["tipo"])

In [ ]:
# Verificar que la ruta existe y la imagen carga
prenda_test = next(p for p in armario if p["id"] == "prenda_004_1")
print(prenda_test)
print("¿Existe el archivo?", os.path.exists(prenda_test["imagen_path"]))